<a href="https://colab.research.google.com/github/GarryCodespace/ANN-Mosfet/blob/main/SmallGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F


# -----------------------
# Hyperparameters
# -----------------------

batch_size = 32
block_size = 64

n_embd = 128
n_head = 4
n_layer = 4

learning_rate = 3e-4
max_iters = 5000

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# -----------------------
# Dataset
# -----------------------

text = """
the cat sat on the mat.
the dog sat on the floor.
the cat saw the dog.
the dog saw the cat.
""" * 1000


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {
    ch: i
    for i, ch in enumerate(chars)
}

itos = {
    i: ch
    for ch, i in stoi.items()
}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return ''.join(itos[i] for i in ids)


data = torch.tensor(
    encode(text),
    dtype=torch.long
)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]


def get_batch(split):

    source = (
        train_data
        if split == "train"
        else val_data
    )

    ix = torch.randint(
        len(source) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        source[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)


# -----------------------
# Attention Head
# -----------------------

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    block_size,
                    block_size
                )
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = (
            q @ k.transpose(-2, -1)
        )

        wei *= (
            k.shape[-1] ** -0.5
        )

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(
            wei,
            dim=-1
        )

        v = self.value(x)

        return wei @ v


# -----------------------
# Multi-Head Attention
# -----------------------

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        num_heads,
        head_size
    ):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

    def forward(self, x):

        out = torch.cat(
            [
                head(x)
                for head in self.heads
            ],
            dim=-1
        )

        return self.proj(out)


# -----------------------
# Feed Forward
# -----------------------

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                n_embd,
                4 * n_embd
            ),
            nn.ReLU(),
            nn.Linear(
                4 * n_embd,
                n_embd
            )
        )

    def forward(self, x):
        return self.net(x)


# -----------------------
# Transformer Block
# -----------------------

class Block(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head
    ):
        super().__init__()

        head_size = (
            n_embd // n_head
        )

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(
            n_embd
        )

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )

    def forward(self, x):

        x = x + self.sa(
            self.ln1(x)
        )

        x = x + self.ffwd(
            self.ln2(x)
        )

        return x


# -----------------------
# GPT
# -----------------------

class MiniGPT(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = (
            nn.Embedding(
                vocab_size,
                n_embd
            )
        )

        self.position_embedding_table = (
            nn.Embedding(
                block_size,
                n_embd
            )
        )

        self.blocks = nn.Sequential(
            *[
                Block(
                    n_embd,
                    n_head
                )
                for _ in range(n_layer)
            ]
        )

        self.ln_f = nn.LayerNorm(
            n_embd
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.shape

        tok_emb = (
            self.token_embedding_table(
                idx
            )
        )

        pos_emb = (
            self.position_embedding_table(
                torch.arange(
                    T,
                    device=device
                )
            )
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(
                B * T,
                C
            )

            targets = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens
    ):

        for _ in range(
            max_new_tokens
        ):

            idx_cond = (
                idx[:, -block_size:]
            )

            logits, _ = self(
                idx_cond
            )

            logits = (
                logits[:, -1, :]
            )

            probs = F.softmax(
                logits,
                dim=-1
            )

            idx_next = (
                torch.multinomial(
                    probs,
                    num_samples=1
                )
            )

            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx


# -----------------------
# Train
# -----------------------

model = MiniGPT().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for step in range(max_iters):

    xb, yb = get_batch("train")

    logits, loss = model(
        xb,
        yb
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

    if step % 500 == 0:
        print(
            f"{step}: "
            f"{loss.item():.4f}"
        )


# -----------------------
# Generate
# -----------------------

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

result = model.generate(
    context,
    max_new_tokens=300
)

print(
    decode(
        result[0].tolist()
    )
)

0: 3.1543
500: 0.0381
1000: 0.0351
1500: 0.0311
2000: 0.0326
2500: 0.0302
3000: 0.0333
3500: 0.0282
4000: 0.0333
4500: 0.0324

the dog sat on the floor.
the cat saw the dog.
the dog saw the cat.

the cat sat on the mat.
the dog sat on the floor.
the cat saw the dog.
the dog saw the cat.

the cat sat on the mat.
the dog sat on the floor.
the cat saw the dog.
the dog saw the cat.

the cat sat on the mat.
the dog sat on the fl
